In [ ]:
# ==== Edit this first ====
REPO_URL = "https://github.com/anshularavind/cs188-default-project.git"
REPO_DIR = "/content/cs188-default-project"

# ==== Training options ====
USE_15M_DIFFUSION = True
EPOCHS = 40
BATCH_SIZE = 64
MAX_EPISODES = None  # e.g. 300 for a faster run

import os
import subprocess
import yaml

def sh(cmd):
    print("+", cmd)
    subprocess.check_call(cmd, shell=True)

# Clone (or refresh) repo
if not os.path.exists(REPO_DIR):
    sh(f"git clone {REPO_URL} {REPO_DIR}")
else:
    sh(f"git -C {REPO_DIR} pull --ff-only")

REPO_ROOT = REPO_DIR
PROJECT_DIR = os.path.join(REPO_ROOT, "cabinet_door_project")
ROBOCASA_DIR = os.path.join(REPO_ROOT, "robocasa")

# 1) Core install (apt + pip + editable installs)
os.chdir(REPO_ROOT)
sh(f"python {PROJECT_DIR}/99_colab_setup.py")

# 2) Kitchen assets + macros (robocasa is outside cabinet_door_project)
# Asset download is large (~10GB) and can take a while.
sh(f"bash -lc \"printf 'y\\n' | python {ROBOCASA_DIR}/robocasa/scripts/download_kitchen_assets.py\"")
sh(f"bash -lc \"printf 'y\\n' | python {ROBOCASA_DIR}/robocasa/scripts/setup_macros.py\"")

# Verify after assets/macros are present
sh(f"python {PROJECT_DIR}/00_verify_installation.py")

# 3) Dataset + augmentation
sh(f"python {PROJECT_DIR}/04_download_dataset.py")
os.chdir(PROJECT_DIR)
sh("python 05b_augment_handle_data.py")

# 4) Optional ~15M diffusion config
config_path = "configs/diffusion_policy.yaml"
if USE_15M_DIFFUSION:
    with open(config_path, "r") as f:
        cfg = yaml.safe_load(f)
    cfg["base_channels"] = 112
    cfg["channel_mults"] = [1, 2, 4]
    cfg["num_res_blocks"] = 2
    cfg["cond_dim"] = 448
    cfg["num_diffusion_iters"] = 100
    cfg["num_inference_iters"] = 16
    config_path = "configs/diffusion_policy_15m_colab.yaml"
    with open(config_path, "w") as f:
        yaml.safe_dump(cfg, f, sort_keys=False)
    print("Using 15M diffusion config:", config_path)

# 5) Train only
train_cmd = f"python 06_train_policy.py --config {config_path} --epochs {EPOCHS} --batch_size {BATCH_SIZE}"
if MAX_EPISODES is not None:
    train_cmd += f" --max_episodes {int(MAX_EPISODES)}"
sh(train_cmd)

print("Training finished. Checkpoints are in /tmp/cabinet_policy_checkpoints")


+ git -C /content/cs188-default-project pull --ff-only
+ python cabinet_door_project/99_colab_setup.py
+ python robocasa/robocasa/scripts/download_kitchen_assets.py


CalledProcessError: Command 'python robocasa/robocasa/scripts/download_kitchen_assets.py' returned non-zero exit status 1.